# Attention Modules

This notebook implements the **attention mechanisms** for the GPT model:

- **ScaledDotProductAttention** — the core attention computation with causal masking.
- **MultiHeadAttention** — runs multiple attention heads in parallel and combines their outputs.

### Scaled Dot-Product Attention

Given query (Q), key (K), and value (V) matrices, attention computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

The scaling factor $\sqrt{d_k}$ prevents the dot products from growing too large,
which would push softmax into regions with tiny gradients.

For **causal (autoregressive) attention**, a mask is applied before softmax so that
each position can only attend to itself and earlier positions — never to the future.
Masked positions are set to $-\infty$ before softmax, which drives their attention
weights to zero.

### Multi-Head Attention

Instead of computing a single attention function, multi-head attention:

1. Projects the input into `num_heads` separate Q, K, V sets (each of dimension `d_k = d_model / num_heads`).
2. Applies scaled dot-product attention independently to each head.
3. Concatenates the head outputs and applies a final linear projection.

This allows the model to attend to information from different representation
subspaces at different positions simultaneously.

In [ ]:
import math

import torch
import torch.nn as nn
from torch import Tensor

## ScaledDotProductAttention

Computes `(Q @ K^T) / sqrt(d_k)`, applies a causal mask to prevent attending
to future positions, applies softmax, then multiplies by V.

The causal mask is an upper-triangular boolean matrix created with `torch.triu`.
Positions where `mask[i][j] = True` (i.e., `j > i`) are set to `-inf` before
softmax, ensuring zero attention weight on future tokens.

| Step | Operation | Shape |
|---|---|---|
| 1 | `Q @ K^T` | `(..., seq_len, seq_len)` |
| 2 | Scale by `1/sqrt(d_k)` | `(..., seq_len, seq_len)` |
| 3 | Apply causal mask (`-inf` for future) | `(..., seq_len, seq_len)` |
| 4 | Softmax over last dim | `(..., seq_len, seq_len)` |
| 5 | `weights @ V` | `(..., seq_len, d_k)` |

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """Scaled dot-product attention with causal masking.

    Computes (Q @ K^T / sqrt(d_k)), applies a causal mask to prevent
    attending to future positions, applies softmax, then multiplies by V.
    """

    def forward(self, Q: Tensor, K: Tensor, V: Tensor, mask: Tensor | None = None) -> Tensor:
        """Compute scaled dot-product attention.

        Args:
            Q: Query tensor of shape (..., seq_len, d_k)
            K: Key tensor of shape (..., seq_len, d_k)
            V: Value tensor of shape (..., seq_len, d_k)
            mask: Optional mask tensor. If None, a causal mask is created.

        Returns:
            Output tensor of the same shape as V.
        """
        d_k = Q.size(-1)
        seq_len = Q.size(-2)

        # Compute attention scores: (Q @ K^T) / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        # Create causal mask if not provided
        if mask is None:
            mask = torch.triu(
                torch.ones(seq_len, seq_len, device=Q.device, dtype=torch.bool),
                diagonal=1,
            )

        # Apply causal mask: set future positions to -inf before softmax
        scores = scores.masked_fill(mask, float("-inf"))

        # Apply softmax and compute weighted sum
        attn_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)

        return output

## MultiHeadAttention

Projects the input into Q, K, V per head, applies `ScaledDotProductAttention`
to each head, concatenates the results, and applies a final linear projection.

The head dimension is `d_k = d_model / num_heads`. If `d_model` is not evenly
divisible by `num_heads`, a `ValueError` is raised at initialization.

| Component | Shape |
|---|---|
| Input `x` | `(batch, seq_len, d_model)` |
| After Q/K/V projection | `(batch, seq_len, d_model)` |
| After reshape to heads | `(batch, num_heads, seq_len, d_k)` |
| After attention per head | `(batch, num_heads, seq_len, d_k)` |
| After concat + output proj | `(batch, seq_len, d_model)` |

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention module.

    Projects input to Q, K, V per head, applies ScaledDotProductAttention
    to each head, concatenates the results, and applies a final linear projection.
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        """Initialize MultiHeadAttention.

        Args:
            d_model: Model dimension.
            num_heads: Number of attention heads.
            dropout: Dropout rate applied after attention weights.

        Raises:
            ValueError: If d_model is not evenly divisible by num_heads.
        """
        super().__init__()

        if d_model % num_heads != 0:
            raise ValueError(
                f"d_model ({d_model}) must be evenly divisible by num_heads ({num_heads})"
            )

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear projections for Q, K, V and output
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor) -> Tensor:
        """Apply multi-head attention.

        Args:
            x: Input tensor of shape (batch, seq_len, d_model).

        Returns:
            Output tensor of shape (batch, seq_len, d_model).
        """
        batch_size, seq_len, _ = x.size()

        # Project to Q, K, V: (batch, seq_len, d_model)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Reshape to (batch, num_heads, seq_len, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # Apply scaled dot-product attention per head
        attn_output = self.attention(Q, K, V)

        # Apply dropout to attention output
        attn_output = self.dropout(attn_output)

        # Concatenate heads: (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        # Final linear projection
        output = self.W_o(attn_output)

        return output

---

## Demos

The cells below demonstrate the attention modules on sample inputs and
visualize the attention weight patterns.

### Demo: ScaledDotProductAttention — Basic Usage

Create random Q, K, V tensors and run them through the attention module.
The output shape should match the input V shape.

In [ ]:
torch.manual_seed(42)

batch, seq_len, d_k = 2, 6, 16
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

sdpa = ScaledDotProductAttention()
output = sdpa(Q, K, V)

print(f"Q shape:      {Q.shape}")
print(f"K shape:      {K.shape}")
print(f"V shape:      {V.shape}")
print(f"Output shape: {output.shape}")
print(f"Shape match:  {output.shape == V.shape}")

### Demo: Visualizing Attention Weights

To see what the causal mask does, we extract the attention weights from
a forward pass and plot them as a heatmap. The upper triangle should be
all zeros — the model never attends to future positions.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib


def get_attention_weights(Q: Tensor, K: Tensor) -> Tensor:
    """Compute causal attention weights (without multiplying by V)."""
    d_k = Q.size(-1)
    seq_len = Q.size(-2)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
    scores = scores.masked_fill(mask, float("-inf"))
    return torch.softmax(scores, dim=-1)


torch.manual_seed(42)
seq_len = 8
d_k = 16
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)

weights = get_attention_weights(Q, K)[0].detach().numpy()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=weights.max())
ax.set_xlabel("Key position")
ax.set_ylabel("Query position")
ax.set_title("Causal Attention Weights")
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
fig.colorbar(im, ax=ax, label="Attention weight")
plt.tight_layout()
plt.show()

print("Upper triangle (future positions) — all zeros:")
for i in range(seq_len):
    for j in range(i + 1, seq_len):
        assert weights[i][j] == 0.0, f"Non-zero at ({i},{j}): {weights[i][j]}"
print("  Verified: all future attention weights are exactly 0.0")

### Demo: Causal Mask Behavior

This demo shows the causal mask in detail. We use an identity-like Q and K
so the raw attention scores are uniform, making the mask effect clearly visible.

With a causal mask:
- Position 0 attends only to position 0 (weight = 1.0)
- Position 1 attends to positions 0–1 (weight ≈ 0.5 each)
- Position `i` attends to positions 0–`i` (weight ≈ `1/(i+1)` each)
- All future positions (`j > i`) get exactly zero weight

In [ ]:
seq_len = 5
d_k = 4

# Use identical Q and K so raw scores are uniform
Q_uniform = torch.ones(1, seq_len, d_k)
K_uniform = torch.ones(1, seq_len, d_k)
V_demo = torch.arange(seq_len, dtype=torch.float).unsqueeze(0).unsqueeze(-1).expand(1, seq_len, d_k)

weights_uniform = get_attention_weights(Q_uniform, K_uniform)[0].detach().numpy()

print("Attention weights with uniform Q, K (causal mask applied):")
print("Each row sums to 1.0, future positions are 0.0\n")
for i in range(seq_len):
    row = [f"{weights_uniform[i][j]:.3f}" for j in range(seq_len)]
    print(f"  Position {i}: [{', '.join(row)}]  sum={weights_uniform[i].sum():.3f}")

print("\nExpected pattern:")
print("  Position 0: attends only to 0 -> weight 1.000")
print("  Position 1: attends to 0,1   -> weight 0.500 each")
print("  Position 2: attends to 0,1,2 -> weight 0.333 each")
print("  ...and so on")

### Demo: MultiHeadAttention — Basic Usage

Create a `MultiHeadAttention` module and pass a sample input through it.
The output shape should match the input shape `(batch, seq_len, d_model)`.

In [ ]:
torch.manual_seed(42)

batch, seq_len, d_model, num_heads = 2, 10, 64, 8
x = torch.randn(batch, seq_len, d_model)

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=0.0)
output = mha(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Shape match:  {output.shape == x.shape}")
print(f"\nd_model={d_model}, num_heads={num_heads}, d_k={d_model // num_heads}")
print(f"Total parameters: {sum(p.numel() for p in mha.parameters()):,}")

### Demo: MultiHeadAttention — Per-Head Attention Weights

Visualize the attention weights from each head. Different heads learn to
attend to different patterns — even with random weights, the patterns vary.

In [ ]:
torch.manual_seed(42)

batch, seq_len, d_model, num_heads = 1, 8, 32, 4
x = torch.randn(batch, seq_len, d_model)

mha_viz = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=0.0)
d_k = d_model // num_heads

# Extract per-head Q, K to compute attention weights
with torch.no_grad():
    Q = mha_viz.W_q(x).view(batch, seq_len, num_heads, d_k).transpose(1, 2)
    K = mha_viz.W_k(x).view(batch, seq_len, num_heads, d_k).transpose(1, 2)

fig, axes = plt.subplots(1, num_heads, figsize=(4 * num_heads, 4))
for head_idx in range(num_heads):
    q_head = Q[0, head_idx].unsqueeze(0)  # (1, seq_len, d_k)
    k_head = K[0, head_idx].unsqueeze(0)
    w = get_attention_weights(q_head, k_head)[0].detach().numpy()

    ax = axes[head_idx]
    ax.imshow(w, cmap="Blues", vmin=0)
    ax.set_title(f"Head {head_idx}")
    ax.set_xlabel("Key pos")
    if head_idx == 0:
        ax.set_ylabel("Query pos")
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))

fig.suptitle("Per-Head Causal Attention Weights", fontsize=14)
plt.tight_layout()
plt.show()

### Demo: Non-Divisible Dimensions Raise ValueError

If `d_model` is not evenly divisible by `num_heads`, the constructor
raises a `ValueError` immediately.

In [ ]:
try:
    bad_mha = MultiHeadAttention(d_model=65, num_heads=8)
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Caught expected error: {e}")